# 08b — integration and final exports

**Feeds:** (superseded for Fig 4d — see stage 12)

**Position in the chain:** ⚠️ Its Fig4e_heatmap_merged_data* outputs render panel 4d and are the SUBMISSION-2 version: that merge has no embryo cells in Floor Plate or NMP. The published Fig 4d comes from stage 12. Kept because it produces other exports and documents the earlier state.

Ported from the original analysis. Saved cell outputs are from the original run, and paths in them appear as `<analysis-root>/...`.

**Changes from the original notebook**, so that it runs from this repository:
1. Paths. The first code cell finds the repository from any working directory inside it; the original looked for `src/` at most two levels up. `PROJECT_ROOT`, which was the analysis directory, is now `scrnaseq/chain_inputs/`, the shipped files that no notebook writes, in the same layout. `DEV_ROOT`, under which the notebook writes, is `$SCRNASEQ_RESULTS_ROOT/trunk_main_dev/` (default `scrnaseq/output/trunk_main_dev/`) instead of the analysis directory's `trunk_main_dev/`.

2. The embryo clusters object. ED Fig 8c needs the earlier human embryo run (46,768 cells, 50 `leiden_embryo` clusters); `adata_embryo_raw` is shared between runs and is not overridden. `SCRNASEQ_INPUT_ROOT` (default `data/scrnaseq_inputs/`) is checked for `earlier_embryo_run/results/intermediates/07_human_embryo/adata_embryo_with_clusters.h5ad`; when absent, the notebook falls back to the run `07_human_embryo` rebuilds here, as before this override existed.

No other line of code was changed.


Depends on: 01_preprocessing, 02_trunk_main, 07_human_embryo.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "trunk_morph_ref").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the repository root containing src/trunk_morph_ref/")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.trunk_morph_ref.paths import chain_inputs_root, scrnaseq_input_root

# Inputs that no notebook writes: scrnaseq/chain_inputs/, laid out as the original analysis directory.
PROJECT_ROOT = chain_inputs_root(REPO_ROOT)
# The earlier human embryo clusters object, fetched from the Zenodo deposit (see data/DOWNLOAD.md).
SCRNASEQ_INPUT_ROOT = scrnaseq_input_root(REPO_ROOT)

import os
os.chdir(REPO_ROOT)


## Setup and Imports


In [ ]:
import matplotlib

matplotlib.rcParams["pdf.fonttype"] = 42
matplotlib.rcParams["ps.fonttype"] = 42

In [ ]:
import pickle
import random

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy
import seaborn as sns
from cmcrameri.cm import batlow
from scipy.sparse import coo_matrix, csr_matrix, load_npz, save_npz

In [ ]:
plt.rcParams["svg.fonttype"] = "none"  # Keep fonts editable in Illustrator

In [ ]:
from src.trunk_morph_ref.cell_ordering import ordered_cells_by_cluster_clustermap
from src.trunk_morph_ref.paths import init_output_paths, scrnaseq_results_root

# Outputs: $SCRNASEQ_RESULTS_ROOT (default scrnaseq/output/), under trunk_main_dev/ as in the original.
DEV_ROOT = scrnaseq_results_root(REPO_ROOT) / "trunk_main_dev"
RESULTS_DIR, MANUSCRIPT_FIG_DIR, EXTENDED_FIG_DIR = init_output_paths(DEV_ROOT)
from src.trunk_morph_ref.aggregation import (
    cluster_averages_sparse_safe,
)
from src.trunk_morph_ref.correlation import (
    gene_corrcoef_sparse_safe,
)
from src.trunk_morph_ref.plotting import (
    bold_selected_heatmap_yticklabels,
    hide_heatmap_axes_and_colorbar,
    keep_only_selected_heatmap_yticklabels,
    plot_empty_heatmap_axes,
)
from src.trunk_morph_ref.preprocessing import (
    assert_gene_id_index,
    contains_symbol,
    ensure_gene_id_index,
    filter_genes_sparse_safe as filter_genes,
    install_scanpy_symbol_defaults,
    normalize_unit_variance_sparse_safe as normalize_unit_variance,
    resolve_symbol_dict,
    resolve_symbols,
    symbols_missing,
    symbols_present,
    var_names_to_symbols,
)

# Use gene symbols for plot labels while keeping `var_names` as stable gene IDs.
install_scanpy_symbol_defaults(sc)

In [ ]:
from src.trunk_morph_ref.pipeline_io import (
    load_h5ad,
    load_npy,
    load_pickle,
    save_h5ad,
    save_json,
    save_npy,
    save_pickle,
    stage_dir,
)

In [ ]:
pre_path = stage_dir(RESULTS_DIR, "01_preprocessing")
trunk_path = stage_dir(RESULTS_DIR, "02_trunk_main")
embryo_path = stage_dir(RESULTS_DIR, "07_human_embryo")

adata_morph_with_clusters = load_h5ad(trunk_path / "adata_morph_with_clusters.h5ad")
integration_gene_panel = load_pickle(trunk_path / "integration_gene_panel.pkl")
fig4_heatmap_label_genes = load_pickle(trunk_path / "key_genes.pkl")
adata_embryo_raw = load_h5ad(pre_path / "adata_embryo_raw.h5ad")

# ED Fig 8c was drawn from the earlier human embryo run (46,768 cells, 50 leiden_embryo
# clusters), not the run this chain rebuilds. That object is in the Zenodo deposit; read it
# from there when present, otherwise fall back to the rebuilt run, as ED Fig 8a and 8b do.
_earlier_embryo_clusters_path = (
    SCRNASEQ_INPUT_ROOT
    / "earlier_embryo_run/results/intermediates/07_human_embryo/adata_embryo_with_clusters.h5ad"
)
if _earlier_embryo_clusters_path.exists():
    adata_embryo_with_clusters = load_h5ad(_earlier_embryo_clusters_path)
else:
    adata_embryo_with_clusters = load_h5ad(embryo_path / "adata_embryo_with_clusters.h5ad")

# Fixed endothelial IDs from stage-02 manual curation; used only for the with/without-endothelial comparison.
lpm_endothelial_cell_ids = [
    "TACGGTACAACACAGG-1",
    "ATCGGCGGTAGGGAGG-1",
    "ACGCACGTCTTAGCAG-1",
    "CTATCTAAGTTCCATG-1",
    "GCATTAGAGAAATTGC-1",
]

for _adata in [
    adata_morph_with_clusters,
    adata_embryo_raw,
    adata_embryo_with_clusters,
]:
    ensure_gene_id_index(_adata)
    assert_gene_id_index(_adata)

print("Loaded upstream intermediates for integration stage (subclustering-independent)")

## Fig. 4e - Key gene expression heatmap for trunk morph and human embryo data

In [ ]:
integration_heatmap_cell_order = []
adata_morph_embryo_gene_subset = adata_morph_with_clusters.copy()
integration_heatmap_cell_order = ordered_cells_by_cluster_clustermap(
    adata=adata_morph_embryo_gene_subset,
    cluster_key="leiden_morph",
    var_names=integration_gene_panel,
    figsize=(40, 40),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="average",
    metric="jensenshannon",
)

In [ ]:
adata_morph_sorted_ = adata_morph_embryo_gene_subset[
    integration_heatmap_cell_order, :
].copy()

In [ ]:
cell_order_embryo = ordered_cells_by_cluster_clustermap(
    adata=adata_embryo_with_clusters,
    cluster_key="leiden_embryo",
    var_names=integration_gene_panel,
    figsize=(40, 40),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="ward",  # perhaps change to method="average" metric="jensenshannon" to be consistent with trunk morph analysis
    metric="euclidean",
)

In [ ]:
adata_embryo_sorted_ = adata_embryo_with_clusters[cell_order_embryo, :].copy()

In [ ]:
sc.pl.heatmap(
    adata_embryo_with_clusters,
    var_names=integration_gene_panel,
    groupby="leiden_embryo",
    figsize=(40, 40),
    vmin=0,
    vmax=5,
    cmap=batlow,
    swap_axes=True,
    show_gene_labels=True,
)

In [ ]:
# HIERARCHIAL CLUSTERING WITHIN EACH CLUSTER
sc.pl.heatmap(
    adata_embryo_sorted_,
    var_names=integration_gene_panel,
    groupby="leiden_embryo",
    figsize=(40, 40),
    vmin=0,
    vmax=5,
    cmap=batlow,
    swap_axes=True,
    show_gene_labels=True,
)

In [ ]:
expanded_cluster_colors = [
    "#1f77b4",  # Forebrain / Midbrain
    "#ff7f0e",  # Hindbrain
    "#279e68",  # Intermediate-Ventral Spinal Cord
    "#d62728",  # Dorsal Spinal Cord
    "#aa40fc",  # Roof Plate
    "#8c564b",  # Neural Crest - Derivatives
    "#a46b5e",  # Neural Crest - Maturing Cranial
    "#73473e",  # Neural Crest 3
    "#e377c2",  # Immature Neuron
    "#b5bd61",  # Posterior Neural Tube
    "#17becf",  # Posterior Neural Tube / Neuromesodermal Progenitors
    "#aec7e8",  # Notochord / Floor Plate
    "#ffbb78",  # Presomitic Mesoderm
    "#98df8a",  # Early Somite
    "#ff9896",  # Mature Somite - Dorsal / Dermomyotome
    "#ff7472",  # Mature Somite - Ventral / Sclerotome
    "#c5b0d5",  # Intermediate Mesoderm
    "#c49c94",  # Lateral Plate Mesoderm - Week 4 Splanchnic
    "#d1a188",  # Lateral Plate Mesoderm - Week 4 Craniofacial / Pharyngeal
    "#b78f7a",  # Lateral Plate Mesoderm 3
    "#a37b6a",  # Lateral Plate Mesoderm 4
    # "#92685c",
    "#f7b6d2",  # Non-Neural Ectoderm - Surface Ectoderm
    "#f99ac4",  # Non-Neural Ectoderm - Cranial Placodal
    "#f57eb5",  # Non-Neural Ectoderm 3
]

# Only analyze/plot embryo clusters that are comparable to those in trunk morph data
adata_embryo_reduced = adata_embryo_sorted_[
    adata_embryo_sorted_.obs.leiden_embryo.isin(
        [
            "Week 4 Forebrain / Midbrain",  # Forebrain / Midbrain
            "Week 4 Hindbrain",  # Hindbrain
            "Week 4 Intermediate-Ventral Spinal Cord",  # Intermediate-Ventral Spinal Cord
            "Week 4 Dorsal Spinal Cord",  # Dorsal Spinal Cord
            "Week 4 Roof Plate",  # Roof Plate
            "Week 3 Neural Crest - Early Migratory",  # Neural Crest
            "Week 4 Neural Crest - Derivatives",
            "Week 4 Neural Crest - Maturing Cranial",
            "Week 4 Neuron - CNS Excitatory",  # Immature Neurons
            "Week 3 Posterior Neural Tube / Neuromesodermal Progenitors",  # Posterior Neural Tube
            "Week 3 Presomitic Mesoderm - Posterior",  # NMPs
            "Week 3 / 4 Notochord",  # Notochord / Floor Plate
            "Week 3 Presomitic Mesoderm - Anterior",  # Presomitic Mesoderm
            "Week 3 Early Somite",  # Early Somite
            "Week 4 Somite - Dorsal / Dermomyotome",  # Mature Somite
            "Week 4 Somite - Ventral / Sclerotome",
            "Week 4 Intermediate Mesoderm / Kidney Progenitors",  # Intermediate Mesoderm
            "Week 3 Lateral Plate Mesoderm - Anterior",  # Lateral Plate Mesoderm
            "Week 4 Lateral Plate Mesoderm - Splanchnic",
            "Week 4 Lateral Plate Mesoderm - Posterior",
            "Week 4 Lateral Plate Mesoderm - Gut Visceral Mesenchyme",
            # "Week 3 / 4 Cardiomyocytes",
            "Week 4 Non-Neural Ectoderm - Surface Ectoderm",  # NNE
            "Week 4 Non-Neural Ectoderm - Cranial Placodal",
            "Week 3 Non-Neural Ectoderm",
        ]
    ),
    :,
]
adata_embryo_reduced.uns["leiden_embryo_colors"] = expanded_cluster_colors

In [ ]:
celltypemapping = {
    "Week 4 Forebrain / Midbrain": "Forebrain / Midbrain",
    "Week 4 Hindbrain": "Hindbrain",
    "Week 4 Intermediate-Ventral Spinal Cord": "Intermediate-Ventral Spinal Cord",
    "Week 4 Dorsal Spinal Cord": "Dorsal Spinal Cord",
    "Week 4 Roof Plate": "Roof Plate",
    "Week 3 Neural Crest - Early Migratory": "Neural Crest",
    "Week 4 Neural Crest - Derivatives": "Neural Crest",
    "Week 4 Neural Crest - Maturing Cranial": "Neural Crest",
    "Week 4 Neuron - CNS Excitatory": "Immature Neuron",
    "Week 3 Posterior Neural Tube / Neuromesodermal Progenitors": "Posterior Neural Tube",
    "Week 3 Presomitic Mesoderm - Posterior": "Presomitic Mesoderm",
    "Week 3 / 4 Notochord": "Notochord / Floor Plate",
    "Week 3 Presomitic Mesoderm - Anterior": "Presomitic Mesoderm",
    "Week 3 Early Somite": "Early Somite",
    "Week 4 Somite - Dorsal / Dermomyotome": "Mature Somite",
    "Week 4 Somite - Ventral / Sclerotome": "Mature Somite",
    "Week 4 Intermediate Mesoderm / Kidney Progenitors": "Intermediate Mesoderm",
    "Week 3 Lateral Plate Mesoderm - Anterior": "Lateral Plate Mesoderm",
    "Week 4 Lateral Plate Mesoderm - Splanchnic": "Lateral Plate Mesoderm",
    "Week 4 Lateral Plate Mesoderm - Posterior": "Lateral Plate Mesoderm",
    "Week 4 Lateral Plate Mesoderm - Gut Visceral Mesenchyme": "Lateral Plate Mesoderm",
    "Week 4 Non-Neural Ectoderm - Surface Ectoderm": "Non-Neural Ectoderm",
    "Week 4 Non-Neural Ectoderm - Cranial Placodal": "Non-Neural Ectoderm",
    "Week 3 Non-Neural Ectoderm": "Non-Neural Ectoderm",
}

In [ ]:
adata_embryo_reduced.obs["leiden_merged"] = [
    celltypemapping[i] for i in adata_embryo_reduced.obs["leiden_embryo"].values
]

In [ ]:
# We subsample n cells from embryo dataset, where n is the number of cells in trunk morph dataset
np.random.seed(0)

# Desired number of cells
n = len(adata_morph_sorted_.obs)

# Subsample
if adata_embryo_reduced.n_obs > n:
    selected_cells = np.random.choice(
        adata_embryo_reduced.obs_names, size=n, replace=False
    )
    adata_embryo_subsampled = adata_embryo_reduced[selected_cells].copy()
else:
    adata_embryo_subsampled = (
        adata_embryo_reduced.copy()
    )  # use full data if n > total cells

In [ ]:
adata_morph_sorted_.obs["leiden_merged"] = adata_morph_sorted_.obs["leiden_morph"]

In [ ]:
adata_merged_ = ad.concat(
    [adata_morph_sorted_, adata_embryo_subsampled],
    join="inner",
    merge="first",
)

# Recreate/validate gene-index metadata after concat so symbol resolution remains available.
ensure_gene_id_index(adata_merged_)
assert_gene_id_index(adata_merged_)


In [ ]:
adata_merged_.obs["leiden_merged"] = pd.Categorical(
    adata_merged_.obs["leiden_merged"],
    categories=adata_morph_with_clusters.obs.leiden_morph.cat.categories,
    ordered=True,
)

In [ ]:
# reorder cells again within clusters, to make substructure more visible
cell_order_merged = ordered_cells_by_cluster_clustermap(
    adata=adata_merged_,
    cluster_key="leiden_merged",
    var_names=integration_gene_panel,
    figsize=(5, 5),
    vmin=0,
    vmax=5,
    cmap=batlow,
    method="average",
    metric="jensenshannon",
)

In [ ]:
adata_merged_sorted_ = adata_merged_[cell_order_merged, :].copy()

In [ ]:
with plt.rc_context({"figure.dpi": (300)}):
    ax = sc.pl.heatmap(
        adata_merged_sorted_,
        var_names=integration_gene_panel,
        groupby="leiden_merged",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        # save="-smd.png",
        show=False,
    )

    bold_selected_heatmap_yticklabels(
        ax, fig4_heatmap_label_genes
    )  # keep bold genes in Illustrator figure (manually delete other gene labels)
    ax["groupby_ax"].set_xlabel("")

    # plt.savefig("alignment_embryo_withaxes.png", bbox_inches="tight")
    plt.show()

In [ ]:
for cell_id in adata_merged_sorted_.obs_names:
    if adata_merged_sorted_.obs.loc[cell_id, "source"] in (["No Bead Morph"]):
        adata_merged_sorted_.obs.loc[cell_id, "tissue"] = "No Bead Morph"

    elif adata_merged_sorted_.obs.loc[cell_id, "source"] in (["BMP4 Bead Morph"]):
        adata_merged_sorted_.obs.loc[cell_id, "tissue"] = "BMP4 Bead Morph"

    else:
        adata_merged_sorted_.obs.loc[cell_id, "tissue"] = "Week 3-4 Human Embryo"

In [ ]:
# 1. Get the list of cell IDs in the same order as in the heatmap
cells_in_order = adata_merged_sorted_.obs_names

# 2. Get the tissue annotation
tissue_labels_in_order = adata_merged_sorted_.obs.loc[cells_in_order, "tissue"]

# 3. Define the color map (make sure all values are included!)
print(tissue_labels_in_order.unique())  # <- use this to verify your keys
source_color_dict = {
    "BMP4 Bead Morph": (1.0, 0.0, 0.0),  # red
    "No Bead Morph": (0.0, 0.0, 1.0),  # blue
    "Week 3-4 Human Embryo": (1.0, 1.0, 0.0),  # white
}

# 4. Map to RGB, fill unknowns with gray or error out
source_color_list = tissue_labels_in_order.map(source_color_dict)

# Check for unmapped entries (will be NaN)
if source_color_list.isnull().any():
    print(
        "⚠️ Unmapped tissue values:",
        tissue_labels_in_order[source_color_list.isnull()].unique(),
    )
    raise ValueError("Please update `source_color_dict` to include all tissue values.")

# 5. Convert to (1, N, 3) array
color_array = np.array(source_color_list.tolist())[np.newaxis, :, :]

# 6. Plot and save
fig, ax = plt.subplots(figsize=(40, 0.5))
ax.imshow(color_array, aspect="auto")
ax.set_xticks([])
ax.set_yticks([])

plt.savefig(
    f"{MANUSCRIPT_FIG_DIR}/Fig4e_heatmap_merged_data_colorbar.png",
    format="png",
    bbox_inches="tight",
    pad_inches=0,
)

In [ ]:
with plt.rc_context({"figure.dpi": (300)}):
    ax = sc.pl.heatmap(
        adata_merged_sorted_,
        var_names=integration_gene_panel,
        groupby="leiden_merged",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        # save="-smd.png",
        show=False,
    )

    bold_selected_heatmap_yticklabels(
        ax, fig4_heatmap_label_genes
    )  # keep bold genes in Illustrator figure (manually delete other gene labels)
    ax["groupby_ax"].set_xlabel("")

    plt.savefig(
        f"{MANUSCRIPT_FIG_DIR}/Fig4e_heatmap_merged_data.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

In [ ]:
with plt.rc_context({"figure.dpi": (300)}):
    ax = plot_empty_heatmap_axes(
        adata_merged_sorted_,
        var_names=integration_gene_panel,
        groupby="leiden_merged",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 40),
        vmin=0,
        vmax=5,
        cmap=batlow,
        # save="-smd.png",
        show=False,
    )

    keep_only_selected_heatmap_yticklabels(ax, fig4_heatmap_label_genes)

    ax["groupby_ax"].set_xlabel("")

    plt.savefig(
        f"{MANUSCRIPT_FIG_DIR}/Fig4e_heatmap_merged_data_axesonly.svg",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

## Supp. Data 2 - Morph-Embryo cluster correlations

In [ ]:
adata_morph_withendo = adata_morph_with_clusters.copy()

if "Endothelial" not in adata_morph_withendo.obs.leiden_morph.cat.categories:
    adata_morph_withendo.obs.leiden_morph = (
        adata_morph_withendo.obs.leiden_morph.cat.add_categories("Endothelial")
    )
adata_morph_withendo.obs.loc[lpm_endothelial_cell_ids, "leiden_morph"] = "Endothelial"


def cluster_averages(adata, cluster_key):
    return cluster_averages_sparse_safe(adata, cluster_key)


avg_morph = cluster_averages(adata_morph_withendo, cluster_key="leiden_morph")
avg_embryo = cluster_averages(adata_embryo_with_clusters, cluster_key="leiden_embryo")

# Resolve the integration panel from symbols to shared gene IDs across objects.
common_genes = resolve_symbols(
    adata_morph_withendo, integration_gene_panel, strict=False, allow_missing=True
)
avg_morph = avg_morph[common_genes]
avg_embryo = avg_embryo[common_genes]

# Compute full correlation matrix
corr_matrix = np.corrcoef(avg_morph.values, avg_embryo.values)

# Extract submatrix: (n_morph_clusters, n_embryo_clusters)
n_morph = avg_morph.shape[0]
corr_submatrix = corr_matrix[:n_morph, n_morph:]

# Format as DataFrame
corr_df = pd.DataFrame(corr_submatrix, index=avg_morph.index, columns=avg_embryo.index)

# FINAL MAPPING:
# Forebrain/Midbrain --> Forebrain - Diencephalon
# Hindbrain --> Hindbrain
# Intermediate-Ventral Spinal Cord --> Intermediate-Ventral Spinal Cord
# Dorsal Spinal Cord --> Dorsal Spinal Cord (MAY CONTAIN SPINAL CORD ROOFPLATE THAT DOWNREGULATED LMX1A)
# Roof Plate --> Roof Plate (BUT NOTE THIS IS LIKELY HINDBRAIN ROOFPLATE)
# Neural Crest --> Neural Crest - Early Migratory + Neural Crest - Derivatives
# Lateral Plate Mesoderm --> Lateral Plate Mesoderm - Week 3 Anterior + Lateral Plate Mesoderm - Week 4 Splanchnic
# + Lateral Plate Mesoderm - Week 4 Posterior +  Lateral Plate Mesoderm - Week 4 Gut Visceral Mesenchyme + Cardiomyocytes
# Non-neural ectoderm --> Non-Neural Ectoderm - Surface Ectoderm+2,
# Intermediate Mesoderm --> Intermediate Mesoderm / Kidney Progenitors
# Somites --> Somites 1, 2
# Notochord / Floor Plate --> Notochord
# Endothelial --> no direct embryo counterpart in this panel


In [ ]:
# All-gene (shared-gene) cluster correlations as orthogonal validation.
shared_gene_ids_all = avg_morph.columns.intersection(avg_embryo.columns)
avg_morph_all = avg_morph[shared_gene_ids_all]
avg_embryo_all = avg_embryo[shared_gene_ids_all]

corr_matrix_all = np.corrcoef(avg_morph_all.values, avg_embryo_all.values)
n_morph_all = avg_morph_all.shape[0]
corr_submatrix_all = corr_matrix_all[:n_morph_all, n_morph_all:]
corr_df_all = pd.DataFrame(
    corr_submatrix_all,
    index=avg_morph_all.index,
    columns=avg_embryo_all.index,
)

with plt.rc_context({'figure.figsize': (9, 8), 'figure.dpi': 300}):
    ax = sns.heatmap(
        corr_df_all.T,
        cmap='seismic',
        vmin=-1,
        vmax=1,
        cbar_kws={'label': 'Pearson r'},
    )
    ax.set_xlabel('Trunk Morph Cluster')
    ax.set_ylabel('Human Embryo Cluster')
    ax.set_title(
        f'All-gene cluster correlations (shared genes, n={len(shared_gene_ids_all)})'
    )
    plt.tight_layout()
    plt.show()


In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 2, "Gene markers and differential expression supporting embryo cell type annotations and trunk morph correspondence"
# Sheet 5, "Morph-Embryo Correlation"
print("Skipped clipboard export for corr_df in automated execution.")


## Calculate DEGs per embryo cluster

In [ ]:
# Calculate top 100 DEGs per cluster, among all genes, by multinomial logistic regression
sc.tl.rank_genes_groups(
    adata_embryo_with_clusters,
    groupby="leiden_embryo",
    method="logreg",
    rankby_abs=False,
    max_iter=1000,
)
DEGs_embryo_noneg = pd.DataFrame(
    adata_embryo_with_clusters.uns["rank_genes_groups"]["names"][0:100]
)
DEGscores_embryo_noneg = pd.DataFrame(
    adata_embryo_with_clusters.uns["rank_genes_groups"]["scores"][0:100]
)

### Supp. Data 2 - Top 100 DEGs and scores per embryo cluster

In [ ]:
# For copy-paste into excel spreadsheet
# Supplementary Data 2, "Gene markers and differential expression supporting embryo cell type annotations and trunk morph correspondence"
# Sheet 4, "Embryo DEGs"
supp_data2_embryo_wide = (
    pd.concat(
        [DEGs_embryo_noneg, DEGscores_embryo_noneg], axis=1, keys=["gene", "score"]
    )
    .swaplevel(0, 1, axis=1)
    .reindex(
        columns=pd.MultiIndex.from_product(
            [DEGs_embryo_noneg.columns, ["gene", "score"]]
        )
    )
)
print("Skipped clipboard export for supp_data2_embryo_wide in automated execution.")


In [ ]:
max_genes_considered = 10
genes_considered = []

for j in range(len(DEGs_embryo_noneg.columns)):
    col = DEGs_embryo_noneg.columns[j]
    ii = 0
    adata_cluster = adata_embryo_raw[
        adata_embryo_with_clusters.obs_names[
            adata_embryo_with_clusters.obs.leiden_embryo == col
        ],
        DEGs_embryo_noneg[col].values,
    ]

    for g in DEGs_embryo_noneg[col].values:
        if ii >= max_genes_considered:
            break
        if g == "GAPDH":
            continue

        # only use gene for cluster where it's DEG score is maximum
        if DEGscores_embryo_noneg[DEGs_embryo_noneg == g].sum().idxmax() == col:
            if (adata_cluster[:, g].X > 2).sum() > 5:
                genes_considered.append(g)
                ii += 1

In [ ]:
# Since we cannot fit all gene labels on the heatmap in a figure, we choose a representative subset of genes to label (including markers for all cell types)
embryo_supp_genes = [
    # Week 4 Optic Field
    "SIX3",
    "RAX",
    "SIX6",
    "ALDH1A3",
    "VAX2",
    "DCT",
    "MITF",
    # Week 4 Forebrain / Midbrain
    "OTX2",
    "PAX6",
    # 'Week 4 Forebrain - Diencephalon / Telencephalon'
    "EMX2",
    "LHX2",
    "NR2E1",
    "BARHL2",
    # 'Week 4 Dorsal Midbrain'
    "WNT2B",
    "WNT4",
    # Week 4 Hindbrain
    "SOX3",
    "HOXA1",
    "HOXB1",
    "FGF17",
    "CRABP1",
    #'Week 3 Anterior Neuroectoderm'
    "SOX2",
    "POU5F1",
    # 'Week 4 Intermediate-Ventral Spinal Cord'
    "NKX6-2",
    "OLIG2",
    "NKX6-1",
    "HES5",
    "HOXA7",
    "HOXD1",
    "HOXC8",
    "HOXD11",
    # 'Week 4 Dorsal Spinal Cord'
    "PAX3",
    "MSX1",
    "MSX2",
    "OLIG3",
    "ZIC2",
    "ZIC1",
    # Week 4 Roof Plate
    "WNT3A",
    # 'Week 3 Neural Crest - Early Migratory'
    "TFAP2A",
    "SNAI2",
    "FOXD3",
    "SOX10",
    "TFAP2B",
    "ERBB3",
    "EDNRB",
    # 'Week 4 Neural Crest - Derivatives'
    "CDH19",
    "PLP1",
    "S100B",
    # 'Week 4 Neural Crest - Maturing Cranial'
    "POSTN",
    "SCRG1",
    # Week 4 Excitatory CNS Neurons
    "HES6",
    "BHLHE22",
    "VXN",
    "KLHDC8A",
    "SRRM4",
    #'Week 4 Neuron - Peripheral Sensory',
    "PPP1R17",
    "NEUROD1",
    "TLX3",
    "POU4F1",
    "NHLH1",
    # 'Week 4 Neuron - Hypothalamus / Neuroendocrine',
    "OTP",
    "NKX2-1",
    "LHX5",
    "POMC",
    "PCSK2",
    # 'Week 4 Neuron - CNS Inhibitory',
    "LHX1",
    "C1QL4",
    "SLC32A1",
    "THSD7A",
    "C1QL2",
    # 'Week 4 Neuron - Autonomic / Cholinergic',
    "PHOX2A",
    "PHOX2B",
    "SLC18A3",
    "CHRNA3",
    "LHX4",
    # 'Week 4 Neuron - Catecholaminergic / Glutamatergic',
    "LHX9",
    "NHLH2",
    "TMEM163",
    "CELF4",
    # 'Week 3 Posterior Neural Tube / Neuromesodermal Progenitors',
    "CDX2",
    "HES7",
    "FGF8",
    "WNT5B",
    "TBXT",
    # 'Week 3 Presomitic Mesoderm - Posterior',
    "TBX6",
    "DLL3",
    "CDX4",
    #'Week 3 / 4 Notochord',
    "SHH",
    "HOPX",
    # 'Week 3 Presomitic Mesoderm - Anterior',
    #'Week 3 Early Somite',
    "RIPPLY1",
    "MEOX1",
    "ALDH1A2",
    # 'Week 4 Somite - Dorsal / Dermomyotome',
    "TCF15",
    "PAX3",
    "UNCX",
    "DMRT2",
    "IGFBP5",
    # 'Week 4 Somite - Ventral / Sclerotome',
    "PAX1",
    "PAX9",
    "NKX3-2",
    "MEOX2",
    "FOXD1",
    "FOXC1",
    # 'Week 4 Intermediate Mesoderm / Kidney Progenitors',
    "PAX8",
    "PAX2",
    "WT1",
    # "Week 3 Intermediate Mesoderm",
    "OSR1",
    "LIX1",
    # 'Week 3 Lateral Plate Mesoderm - Anterior',
    "TMEM88",
    "HAND1",
    "BMP4",
    # 'Week 3 Lateral Plate Mesoderm - Craniofacial / Pharyngeal',
    "SIX1",
    "TGFBI",
    "SDC2",
    "DPT",
    # 'Week 4 Lateral Plate Mesoderm - Splanchnic',
    "GATA4",
    "TCF21",
    "TBX18",
    "HGF",
    "MSC",
    # 'Week 4 Lateral Plate Mesoderm - Craniofacial / Pharyngeal',
    "DLX5",
    "DLX6",
    "PRRX2",
    "PDGFRA",
    "SNAI1",
    "HOXD11",
    # 'Week 4 Lateral Plate Mesoderm - Posterior',
    "PITX1",
    "GATA6",
    "HAND2",
    "COL6A3",
    "TBX3",
    # 'Week 4 Lateral Plate Mesoderm - Gut Visceral Mesenchyme',
    "BARX1",
    "HLX",
    "IGF2",
    "GPC3",
    "CDKN1C",
    # 'Week 3 / 4 Cardiomyocytes',
    "MYL7",
    "MYH6",
    "HSPB7",
    "TNNT2",
    "NKX2-5",
    #'Week 4 Skeletal Myocytes',
    "KLHL41",
    "MYLPF",
    "TNNC2",
    "NEB",
    "MYF6",
    # 'Week 3 Endothelial',
    "CD34",
    "ESAM",
    "PROCR",
    "ECSCR",
    #'Week 4 Endothelial',
    "SOX18",
    "KDR",
    "CDH5",
    "FLT1",
    "EGFL7",
    "TIE1",
    # 'Week 4 Head Mesenchyme - Multipotent Progenitors',
    "ALCAM",
    "SIX2",
    "LUM",
    "COL9A2",
    "CPED1",
    # 'Week 4 Head Mesenchyme - Pharyngeal Arch Core Mesoderm',
    "CNMD",
    "CYP1B1",
    "DAB2",
    "MIR99AHG",
    "CXCL12",
    # 'Week 4 Head Mesenchyme - Frontonasal Mesoderm',
    "ALX1",
    "RGCC",
    "ALX4",
    "EYA4",
    "FHL2",
    # 'Week 4 Head Mesenchyme - First Arch Oral / Palatal Mesenchyme',
    "PRRX1",
    "FRZB",
    "LHX8",
    "DLX2",
    "DLX1",
    # 'Week 4 Head Mesenchyme - Branchiomeric Muscle Progenitors',
    "PITX2",
    "DKK2",
    "RARB",
    "FLRT3",
    "FOXP1",
    # 'Week 4 Head Mesenchyme - Forebrain-MULTIPLET',
    # 'Week 4 Trunk Mesenchyme',
    # marked by mesenchyme markers and HOX genes already in list
    # 'Week 4 Non-Neural Ectoderm - Surface Ectoderm'
    "MIR205HG",
    "WNT6",
    "TACSTD2",
    "TP63",
    "KRT19",
    # 'Week 4 Non-Neural Ectoderm - Cranial Placodal',
    "SIX1",  # already used for cranial LPM
    "EPCAM",
    "ESRP1",
    "CLDN7",
    "MARVELD3",
    "RAB25",
    # 'Week 3 Non-Neural Ectoderm',
    "CLDN6",
    "KRT23",
    "PRSS8",
    "SOX15",
    "SPINT1",
    # 'Week 4 Definitive Endoderm - Fetal Liver',
    "AFP",
    "SERPINA1",
    "FABP1",
    "APOA2",
    "FGG",
    # 'Week 4 Definitive Endoderm - Intestinal',
    "HMGCS2",
    "LINC00261",
    "KYNU",
    "FOXA1",
    "RFX6",
    # 'Week 3 / 4 Erythroid Precursors',
    "AHSP",
    "HBE1",
    "HBG1",
    "ALAS2",
    "GYPA",
    # 'Week 3 / 4 Hematopoietic Progenitors'
    "ARHGDIB",
    "FERMT3",
    "FCER1G",
    "SPI1",
    "PLEK",
]

In [ ]:
# Due to figure space issues, we cut down to 90 genes, shown below:
embryo_supp_genes = [
    "RAX",
    "SIX3",
    "ALDH1A3",
    "OTX2",
    "EMX2",
    "LHX2",
    "WNT4",
    "HOXB1",
    "HOXA1",
    "SOX2",
    "NKX6-1",
    "OLIG2",
    "MSX1",
    "OLIG3",
    "WNT3A",
    "SOX10",
    "FOXD3",
    "PLP1",
    "S100B",
    "POSTN",
    "BHLHE22",
    "SRRM4",
    "POU4F1",
    "TLX3",
    "OTP",
    "POMC",
    "SLC32A1",
    "LHX1",
    "PHOX2B",
    "SLC18A3",
    "LHX9",
    "NHLH2",
    "TBXT",
    "CDX2",
    "TBX6",
    "DLL3",
    "SHH",
    "RIPPLY1",
    "MEOX1",
    "ALDH1A2",
    "DMRT2",
    "TCF15",
    "PAX1",
    "NKX3-2",
    "PAX8",
    "WT1",
    "OSR1",
    "HAND1",
    "BMP4",
    "DLX5",
    "PRRX2",
    "GATA4",
    "TCF21",
    "PITX1",
    "HAND2",
    "HOXD11",
    "BARX1",
    "HLX",
    "TNNT2",
    "NKX2-5",
    "MYLPF",
    "MYF6",
    "PROCR",
    "ESAM",
    "KDR",
    "CDH5",
    "SIX2",
    "DAB2",
    "CXCL12",
    "ALX1",
    "ALX4",
    "LHX8",
    "DLX2",
    "PITX2",
    "FOXP1",
    "RGCC",
    "TP63",
    "TACSTD2",
    "SIX1",
    "CLDN7",
    "CLDN6",
    "SOX15",
    "AFP",
    "SERPINA1",
    "RFX6",
    "HMGCS2",
    "AHSP",
    "GYPA",
    "SPI1",
    "ARHGDIB",
]

### ED Fig. 8c - Embryo cluster DEG expression heatmap

In [ ]:
# Save png file without axes for low file size import into Illustrator
with plt.rc_context({"figure.dpi": (300)}):
    fig_dict = sc.pl.heatmap(
        adata_embryo_sorted_,
        var_names=genes_considered,
        groupby="leiden_embryo",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 60),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )

    hide_heatmap_axes_and_colorbar(fig_dict)

    # # Get the current ytick labels (these are the gene names)
    # tick_labels = [label.get_text() for label in ax["heatmap_ax"].get_yticklabels()]

    # # Replace any label not in key_genes with an empty string
    # new_labels = [gene if gene in embryo_supp_genes else "" for gene in tick_labels]

    # # Apply the new label list
    # ax["heatmap_ax"].set_yticklabels(new_labels)
    # ax["groupby_ax"].set_xlabel("")

    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8c_heatmap_top10degs_embryo_noaxes.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

In [ ]:
# save svg file without data for editable import into Illustrator
with plt.rc_context({"figure.dpi": (300)}):
    ax = plot_empty_heatmap_axes(
        adata_embryo_sorted_,
        var_names=genes_considered,
        groupby="leiden_embryo",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 60),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )

    keep_only_selected_heatmap_yticklabels(ax, embryo_supp_genes)
    ax["groupby_ax"].set_xlabel("")

    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8c_heatmap_top10degs_embryo_axesonly.svg",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

In [ ]:
# Plot png for visual inspection
with plt.rc_context({"figure.dpi": (300)}):
    ax = sc.pl.heatmap(
        adata_embryo_sorted_,
        var_names=genes_considered,
        groupby="leiden_embryo",
        swap_axes=True,
        show_gene_labels=True,
        figsize=(40, 60),
        vmin=0,
        vmax=5,
        cmap=batlow,
        show=False,
        var_group_labels=[],
    )

    keep_only_selected_heatmap_yticklabels(ax, embryo_supp_genes)
    ax["groupby_ax"].set_xlabel("")

    plt.savefig(
        f"{EXTENDED_FIG_DIR}/EDFig8c_heatmap_top10degs_embryo.png",
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()

## Save Stage Outputs
Persist integrated objects and stage metadata.


In [ ]:
stage_path = stage_dir(RESULTS_DIR, "08b_integration_and_final_exports")
ensure_gene_id_index(adata_merged_sorted_)
assert_gene_id_index(adata_merged_sorted_)
save_h5ad(adata_merged_sorted_, stage_path / "adata_merged_sorted_.h5ad")
save_json(
    {"stage": "08b_integration_and_final_exports", "seed_policy": "all seeds set to 0"},
    stage_path / "meta.json",
)
print(f"Saved integration intermediates to {stage_path}")